> **Note:** This notebook reuses patterns inspired by the Azure AI Gateway Labs (https://github.com/Azure-Samples/AI-Gateway).

# 🚀 Citadel Hosted Agent - Build, Deploy & Test

## Overview

This notebook builds a **Foundry Hosted Agent** for a Contoso HR scenario, governed end-to-end with the **Microsoft Agent Governance Toolkit (AGT)**, deploys it to the Foundry project in the spoke resource group, and tests it end-to-end.

| Component | Detail |
|-----------|--------|
| **Agent Framework** | Microsoft Agent Framework (`agent-framework`) |
| **Governance** | AGT MAF adapter (`agent_os.integrations.maf_adapter`) — policy, capability guard, audit trail, rogue detection |
| **Protocol** | Responses API (hosted on port 8088) |
| **Model** | `gpt-4.1` via APIM BYO Gateway connection (`Hub-HR-ChatAgent-DEV-LLM/gpt-4.1`) |
| **Tools** | `get_pto_balance`, `get_holiday_schedule`, `get_benefits_summary`, `get_open_enrollment_window` |
| **Policy** | YAML file bundled in the container (`policies/hr-policy.yaml`) — denies SSN disclosure and 3rd-party salary lookups, audits PTO/benefits queries |
| **Telemetry** | AGT OpenTelemetry (`enable_otel`) → Application Insights. Emits AGT policy spans (`agt.policy.*`) and metrics (`agt.policy.evaluations`, `agt.policy.denials`, `agt.policy.latency_ms`). |
| **Container Build** | `az acr build` (cloud build — no local Docker required) |

## Prerequisites

- Citadel Governance Hub deployed (`azd up` completed)
- Spoke Foundry resources deployed (`workshop/scripts/deploy-spoke-foundry.ps1` or `.sh`)
- BYO Gateway connection `Hub-HR-ChatAgent-DEV-LLM` created (notebook 3)

- Python packages installed via `uv sync` in the `workshop/` directory- Azure CLI logged in (`az login`)

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Read environment values from the `azd` environment and configure variables for this notebook.

In [ ]:
# Read environment values from azd
import subprocess, json

def azd_get_value(key):
    """Read a value from the current azd environment."""
    result = subprocess.run(
        ["azd", "env", "get-value", key],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to read {key}: {result.stderr.strip()}")
    return result.stdout.strip()

env_governance_hub_resource_group = azd_get_value("AZURE_RESOURCE_GROUP")
env_location = azd_get_value("AZURE_LOCATION")

env_foundry_subscription_id = azd_get_value("AZURE_SUBSCRIPTION_ID")
env_foundry_resource_group  = azd_get_value("SPOKE_RESOURCE_GROUP")
env_foundry_account_name    = azd_get_value("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
env_foundry_project_name    = azd_get_value("SPOKE_AI_FOUNDRY_PROJECT_NAME")
env_acr_name                = azd_get_value("SPOKE_ACR_NAME")
env_acr_login_server        = azd_get_value("SPOKE_ACR_LOGIN_SERVER")

print(f"Citadel Hub resource group: {env_governance_hub_resource_group}")
print(f"Location: {env_location}")
print(f"Foundry: {env_foundry_account_name} (Project: {env_foundry_project_name})")
print(f"Spoke RG: {env_foundry_resource_group}")
print(f"ACR: {env_acr_name} ({env_acr_login_server})")

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils

# ============================================================================
# 🔧 HOSTED AGENT CONFIGURATION
# To deploy a new version: bump hosted_agent_image_tag (e.g. "v1" → "v2"),
# then re-run from step 2 onward. The agent identity and RBAC persist
# across versions — only the image tag needs to change.
# ============================================================================
hosted_agent_name       = "citadel-hr-agent"
hosted_agent_image_tag  = "v1"
hosted_agent_model      = "Hub-HR-ChatAgent-DEV-LLM/gpt-4.1"  # BYO Gateway connection/model

# Derived values
hosted_agent_image      = f"{env_acr_login_server}/{hosted_agent_name}:{hosted_agent_image_tag}"
foundry_project_endpoint = f"https://{env_foundry_account_name}.services.ai.azure.com/api/projects/{env_foundry_project_name}"

utils.print_info(f"Agent name: {hosted_agent_name}")
utils.print_info(f"Image: {hosted_agent_image}")
utils.print_info(f"Model: {hosted_agent_model}")
utils.print_info(f"Foundry endpoint: {foundry_project_endpoint}")

<a id='1'></a>
### 1️⃣ Verify Azure CLI Login

Ensure you are logged in to Azure CLI and the correct subscription is active.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Generate Hosted Agent Source Files

This cell writes the agent application code into `workshop/hosted-agent/`. The agent is a **Contoso HR Assistant** governed end-to-end by the **Agent Governance Toolkit (AGT)** via the official `agent_os.integrations.maf_adapter`.

**HR tools** (all dummy / deterministic — no external systems needed):

- **`get_pto_balance(employee_id)`** — Remaining + used PTO days and accrual rate
- **`get_holiday_schedule(country)`** — Next 3 company holidays for `US` / `UK` / `JP`
- **`get_benefits_summary(plan_type)`** — Summary for `medical` / `dental` / `vision` / `401k`
- **`get_open_enrollment_window()`** — Simulated enrollment window dates
- **`delete_user(user_id)`** — Simulated destructive HR admin action, included to demonstrate policy blocking before a dangerous tool can run

**AGT governance** is wired into MAF via `Agent(..., middleware=...)`. The hosted-agent streaming path uses a small streaming-compatible policy wrapper around AGT's `PolicyEvaluator` so deny decisions return a `ResponseStream` instead of a non-stream `AgentResponse`. Four layers run on every request:

1. `AuditTrailMiddleware` — logs every agent invocation
2. `GovernancePolicyMiddleware` — evaluates the YAML policy in `policies/hr-policy.yaml` (denies SSN disclosure, 3rd-party salary lookups, and user deletion/deactivation requests; audits PTO/benefits queries)
3. `CapabilityGuardMiddleware` — restricts tool calls to the 5 HR tools above
4. `RogueDetectionMiddleware` — behavioural anomaly detection. Uses `agent-sre`; the adapter falls back gracefully via `try/except ImportError` if the package isn't installable in the container.

The agent uses the **BYO Gateway connection** (`Hub-HR-ChatAgent-DEV-LLM/gpt-4.1`) to access the LLM through APIM — it does **not** deploy any models locally to the Foundry project.

**OpenTelemetry**: the Foundry hosted-agent platform auto-configures Azure Monitor via the `microsoft-opentelemetry` distro using the injected `APPLICATIONINSIGHTS_CONNECTION_STRING`. `main.py` does **not** re-configure OTel (doing so triggers `Overriding of current TracerProvider is not allowed` warnings and leaves AGT spans attached to a dead local provider). Instead, AGT telemetry flows through the canonical surface: a shared `AuditLog` is passed into `create_governance_middleware(audit_log=...)`, and its `AuditLog.export_cloudevents()` output is picked up by the global OTel `LoggerProvider`. `agent.yaml` sets `OTEL_RESOURCE_ATTRIBUTES` so AGT events appear under `service.name=citadel-hr-agent` in Application Insights. Do not set `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true` for this hosted-agent pattern: the Azure AI Projects experimental Responses wrapper returns an `AsyncStreamWrapper` without the raw `.headers` MAF expects, which breaks streamed tool-call responses in the playground. MAF 1.6 / microsoft-opentelemetry still provides the supported OpenAI/HTTP instrumentation.

> **Note:** The `RBAC` requirement for `az acr build` is **Contributor** on the ACR or the spoke resource group. The Foundry project managed identity needs **AcrPull** on the ACR (assigned by `deploy-spoke-foundry` script).


In [ ]:
import pathlib

agent_dir = pathlib.Path("hosted-agent")
agent_dir.mkdir(exist_ok=True)
policies_dir = agent_dir / "policies"
policies_dir.mkdir(exist_ok=True)

# ---- main.py ----
main_py = r'''import os
from datetime import date, datetime, timedelta, timezone as tz
from typing import Annotated

import logging
import threading
import time as _time

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential
from opentelemetry import trace
from pydantic import Field

# ---------------------------------------------------------------------------
# HR tools (all dummy / deterministic — no external systems).
# ---------------------------------------------------------------------------

@tool(approval_mode="never_require")
def get_pto_balance(
    employee_id: Annotated[str, Field(description="Employee ID, e.g. 'E12345'")]
) -> str:
    """Get the remaining PTO balance, days used this year, and accrual rate for an employee."""
    import hashlib
    seed = int(hashlib.md5(employee_id.encode()).hexdigest()[:6], 16)
    accrued = 20 + (seed % 6)           # 20-25 days/yr
    used = seed % accrued
    remaining = accrued - used
    return (
        f"PTO for employee {employee_id}: {remaining} days remaining "
        f"({used} used / {accrued} accrued this year, accrual rate "
        f"{accrued/12:.2f} days/month)."
    )


@tool(approval_mode="never_require")
def get_holiday_schedule(
    country: Annotated[str, Field(description="ISO country code: 'US', 'UK', or 'JP'")]
) -> str:
    """Get the next 3 upcoming Contoso company holidays for a country."""
    today = date.today()
    catalog = {
        "US": [("Memorial Day", (5, 27)), ("Independence Day", (7, 4)),
               ("Labor Day", (9, 2)), ("Thanksgiving", (11, 28)),
               ("Christmas Day", (12, 25))],
        "UK": [("Spring Bank Holiday", (5, 27)), ("Summer Bank Holiday", (8, 26)),
               ("Christmas Day", (12, 25)), ("Boxing Day", (12, 26))],
        "JP": [("Showa Day", (4, 29)), ("Greenery Day", (5, 4)),
               ("Marine Day", (7, 15)), ("Mountain Day", (8, 11)),
               ("Culture Day", (11, 3))],
    }
    code = country.upper()
    if code not in catalog:
        return f"No holiday schedule configured for country '{country}'. Supported: US, UK, JP."
    upcoming = []
    for name, (m, d) in catalog[code]:
        candidate = date(today.year, m, d)
        if candidate < today:
            candidate = date(today.year + 1, m, d)
        upcoming.append((candidate, name))
    upcoming.sort()
    top3 = ", ".join(f"{name} ({d.isoformat()})" for d, name in upcoming[:3])
    return f"Next 3 Contoso holidays in {code}: {top3}."


@tool(approval_mode="never_require")
def get_benefits_summary(
    plan_type: Annotated[str, Field(description="One of: 'medical', 'dental', 'vision', '401k'")]
) -> str:
    """Get a high-level summary of a Contoso benefits plan."""
    summaries = {
        "medical":  "Medical (Contoso PPO): $250 deductible, $20 PCP copay, 90% in-network coinsurance, $3,000 OOP max.",
        "dental":   "Dental (Contoso Premier): 100% preventive, 80% basic, 50% major; $2,000 annual maximum.",
        "vision":   "Vision (Contoso View): annual eye exam $0, $200 frames allowance every 12 months.",
        "401k":     "401(k): Contoso matches 100% of the first 6% of eligible pay; immediate vesting; Roth + traditional options.",
    }
    key = plan_type.lower()
    if key not in summaries:
        return f"Unknown plan '{plan_type}'. Supported: medical, dental, vision, 401k."
    return summaries[key]


@tool(approval_mode="never_require")
def get_open_enrollment_window() -> str:
    """Get the current open enrollment window dates for Contoso benefits."""
    today = date.today()
    start = date(today.year, 11, 1)
    end = date(today.year, 11, 21)
    if today > end:
        start = date(today.year + 1, 11, 1)
        end = date(today.year + 1, 11, 21)
    days = (start - today).days
    when = "currently open" if start <= today <= end else f"opens in {days} days"
    return f"Contoso open enrollment: {start.isoformat()} → {end.isoformat()} ({when})."


@tool(approval_mode="never_require")
def delete_user(
    user_id: Annotated[str, Field(description="Employee/user ID to delete, e.g. 'E12345'")]
) -> str:
    """Simulate deleting a Contoso HR user account."""
    return f"SIMULATION ONLY: user {user_id} would be deleted."


# ---------------------------------------------------------------------------
# OpenTelemetry note.
#
# The Foundry hosted-agent platform auto-configures the global OTel
# TracerProvider / MeterProvider / LoggerProvider via the
# microsoft-opentelemetry distro using the injected
# APPLICATIONINSIGHTS_CONNECTION_STRING. We deliberately do NOT call
# configure_azure_monitor() or agentmesh.governance.enable_otel() here:
# both attempt to override the global providers and trigger
# "Overriding of current TracerProvider is not allowed" warnings, which
# leaves AGT spans attached to a process-local provider that nobody
# exports.
#
# AGT telemetry instead flows through the canonical
# AuditLog.export_cloudevents() surface: we construct a shared AuditLog,
# pass it to create_governance_middleware(audit_log=...), and the existing
# OTel pipeline picks it up. Custom span emission can use:
#     tracer = trace.get_tracer(__name__)
# ---------------------------------------------------------------------------

tracer = trace.get_tracer(__name__)


# ---------------------------------------------------------------------------
# AGT AuditLog -> OTel logger flush task.
#
# create_governance_middleware records every policy / capability / rogue
# decision into the in-memory AuditLog. To surface those decisions in
# Application Insights we periodically drain AuditLog.export_cloudevents()
# onto a stdlib logger. The platform distro auto-bridges stdlib logging to
# the OTel LoggerProvider, so each CloudEvent becomes a structured log
# record (visible in App Insights "traces" table with logger name
# `agt.audit`).
#
# Runs in a daemon thread so it does not interfere with the Hypercorn
# event loop owned by ResponsesHostServer.run().
# ---------------------------------------------------------------------------

def _start_audit_flusher(audit_log, interval_s: float = 5.0) -> None:
    audit_logger = logging.getLogger("agt.audit")
    # Force INFO so flusher diagnostics survive any root-level filter the
    # platform may install. logger.propagate stays True so the OTel logging
    # bridge installed by the distro still picks the records up.
    audit_logger.setLevel(logging.INFO)

    # One-shot startup diagnostic: log the actual AuditLog API surface so we
    # can see in stderr which method name to call (export_cloudevents vs.
    # export vs. something else).
    public_methods = sorted(m for m in dir(audit_log) if not m.startswith("_"))
    audit_logger.info(
        "agt.audit flusher starting (interval=%.1fs); AuditLog API: %s",
        interval_s, ", ".join(public_methods),
    )

    def _flush_loop() -> None:
        # AGT 3.6.0 AuditLog.export_cloudevents() returns the FULL ordered
        # list of events recorded so far (no incremental kwarg). We track how
        # many we've already emitted and slice from there each tick.
        seen = 0
        tick = 0
        while True:
            tick += 1
            drained_this_tick = 0
            err: Exception | None = None
            try:
                events = audit_log.export_cloudevents() or []
                new_events = events[seen:]
                for ce in new_events:
                    ce_type = ce.get("type") if isinstance(ce, dict) else getattr(ce, "type", None)
                    audit_logger.info(
                        "agt.audit.event",
                        extra={"agt.event.type": ce_type, "agt.cloudevent": ce},
                    )
                    drained_this_tick += 1
                seen = len(events)
            except Exception as exc:  # never let the flusher die silently
                err = exc

            # Heartbeat once per minute (every 12 ticks at 5s interval) so we
            # can confirm the thread is alive even when nothing is drained.
            if err is not None:
                audit_logger.warning(
                    "agt.audit flush tick %d failed: %s: %s",
                    tick, type(err).__name__, err,
                )
            elif drained_this_tick > 0:
                audit_logger.info(
                    "agt.audit flush tick %d drained=%d total=%d",
                    tick, drained_this_tick, seen,
                )
            elif tick % 12 == 1:
                audit_logger.info(
                    "agt.audit flush tick %d heartbeat (no new events; total=%d)",
                    tick, seen,
                )
            _time.sleep(interval_s)

    threading.Thread(
        target=_flush_loop,
        name="agt-audit-flusher",
        daemon=True,
    ).start()


def _create_streaming_governance_middleware(
    *,
    policy_directory: str,
    allowed_tools: list[str],
    agent_id: str,
    audit_log,
) -> list:
    from agent_framework import (
        AgentMiddleware,
        AgentResponse,
        AgentResponseUpdate,
        Content,
        Message,
        ResponseStream,
    )
    from agent_os.integrations.maf_adapter import create_governance_middleware
    from agent_os.policies import PolicyEvaluator

    class StreamingGovernancePolicyMiddleware(AgentMiddleware):
        def __init__(self, evaluator: PolicyEvaluator, audit_log) -> None:
            self.evaluator = evaluator
            self.audit_log = audit_log

        async def process(self, context, call_next) -> None:
            agent_name = getattr(context.agent, "name", "unknown")
            messages = getattr(context, "messages", None) or []
            last_message_text = ""
            if messages:
                last_msg = messages[-1]
                last_message_text = getattr(last_msg, "text", None) or str(last_msg)

            decision = self.evaluator.evaluate(
                {
                    "agent": agent_name,
                    "message": last_message_text,
                    "timestamp": _time.time(),
                    "stream": getattr(context, "stream", False),
                    "message_count": len(messages),
                }
            )

            metadata = getattr(context, "metadata", {})
            metadata["governance_decision"] = decision

            if decision.allowed:
                if self.audit_log:
                    self.audit_log.log(
                        event_type="policy_evaluation",
                        agent_did=agent_name,
                        action="allow",
                        data={
                            "matched_rule": decision.matched_rule,
                            "message_preview": last_message_text[:200],
                        },
                        outcome="success",
                        policy_decision=decision.action,
                    )
                await call_next()
                return

            logging.getLogger("agent_os.integrations.maf_adapter").info(
                "Policy DENY for agent '%s': %s (rule=%s)",
                agent_name,
                decision.reason,
                decision.matched_rule,
            )
            refusal = f"Policy violation: {decision.reason}"

            if self.audit_log:
                self.audit_log.log(
                    event_type="policy_violation",
                    agent_did=agent_name,
                    action="deny",
                    data={
                        "reason": decision.reason,
                        "matched_rule": decision.matched_rule,
                        "message_preview": last_message_text[:200],
                    },
                    outcome="denied",
                    policy_decision=decision.action,
                )

            if getattr(context, "stream", False):
                async def _deny_stream():
                    yield AgentResponseUpdate(
                        contents=[Content.from_text(refusal)],
                        role="assistant",
                        agent_id=agent_name,
                    )

                context.result = ResponseStream(
                    _deny_stream(),
                    finalizer=AgentResponse.from_updates,
                )
            else:
                context.result = AgentResponse(
                    messages=[Message("assistant", [refusal])]
                )

    evaluator = PolicyEvaluator()
    evaluator.load_policies(policy_directory)
    stack = create_governance_middleware(
        policy_directory=None,
        allowed_tools=allowed_tools,
        agent_id=agent_id,
        enable_rogue_detection=True,
        audit_log=audit_log,
    )
    stack.insert(1, StreamingGovernancePolicyMiddleware(evaluator, audit_log))
    return stack


# ---------------------------------------------------------------------------
# Main entrypoint.
# ---------------------------------------------------------------------------

def main() -> None:
    from agentmesh.governance import AuditLog

    tools = [
        get_pto_balance,
        get_holiday_schedule,
        get_benefits_summary,
        get_open_enrollment_window,
        delete_user,
    ]
    allowed_tool_names = [t.name if hasattr(t, "name") else t.__name__ for t in tools]

    agent_name_env = os.environ.get("OTEL_SERVICE_NAME", "citadel-hr-agent")

    # Shared AuditLog — emits OTel-compatible CloudEvents via
    # audit_log.export_cloudevents(); plugs into the already-configured
    # global LoggerProvider installed by the platform distro.
    audit_log = AuditLog()

    governance = _create_streaming_governance_middleware(
        policy_directory="policies",
        allowed_tools=allowed_tool_names,
        agent_id=agent_name_env,
        audit_log=audit_log,
    )

    # Drain AGT audit events into the OTel pipeline every 5s.
    _start_audit_flusher(audit_log, interval_s=5.0)

    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=DefaultAzureCredential(),
    )

    agent = Agent(
        client=client,
        name=agent_name_env,
        instructions=(
            "You are Contoso HR Assistant, a friendly internal HR helper. "
            "Use the provided tools to answer questions about an employee's own PTO balance, "
            "Contoso company holidays, benefits plans (medical/dental/vision/401k), and the "
            "open enrollment window. The delete_user tool exists only to demonstrate governance "
            "for explicit user deletion requests. Never disclose another employee's personal data "
            "(salary, SSN, compensation). Be concise."
        ),
        tools=tools,
        middleware=governance,
        # History is managed by the hosting infrastructure.
        default_options={"store": False},
    )

    server = ResponsesHostServer(agent)
    server.run()


if __name__ == "__main__":
    main()
'''
(agent_dir / "main.py").write_text(main_py.strip(), encoding="utf-8")
utils.print_ok("Created hosted-agent/main.py")

# ---- requirements.txt ----
# agent-sre is optional — the AGT MAF adapter falls back gracefully via
# try/except ImportError if it isn't installable. Keeping it in for the
# full demo (rogue-detection middleware).
requirements_txt = """\
# Pinned to the verified-good set from the foundry-hosted-agents skill.
# The umbrella `agent-framework` meta-package resolves to versions with
# the AsyncStreamWrapper.parse() regression that breaks streamed tool
# calls in the hosted-agent server.
agent-framework-core~=1.6.0
agent-framework-foundry~=1.6.0
agent-framework-foundry-hosting==1.0.0a260521
mcp~=1.10.0
pydantic==2.12.5
azure-identity>=1.19.0,<1.26.0a0
agent-os-kernel
agentmesh-platform
agent-sre
"""
(agent_dir / "requirements.txt").write_text(requirements_txt, encoding="utf-8")
utils.print_ok("Created hosted-agent/requirements.txt")

# ---- Dockerfile ----
# Use uv for fast, reliable installs (10-100x faster than pip). The official
# uv image provides the binary; we copy it into a slim Python base.
dockerfile = """\
FROM python:3.13-slim

# Install uv from the official distroless image
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/

ENV UV_SYSTEM_PYTHON=1 \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /app
COPY requirements.txt .
RUN uv pip install --system --no-cache --prerelease=if-necessary-or-explicit -r requirements.txt
RUN python -c "from pydantic._internal._typing_extra import eval_type_backport; import mcp; from agent_framework_foundry_hosting import ResponsesHostServer; print('dependency smoke ok')"

COPY main.py .
COPY policies/ ./policies/

EXPOSE 8088

CMD ["python", "main.py"]
"""
(agent_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")
utils.print_ok("Created hosted-agent/Dockerfile")

# ---- policies/hr-policy.yaml ----
hr_policy_yaml = """\
name: contoso-hr-policy
version: "1.0"
description: Governance policy for the Contoso HR hosted agent.

defaults:
  action: allow

rules:
  - name: block-ssn-disclosure
    condition:
      field: message
      operator: matches
      value: "\\\\b\\\\d{3}-\\\\d{2}-\\\\d{4}\\\\b"
    action: deny
    message: "SSN detected in request — blocked for PII protection."
    priority: 1000

  - name: block-user-deletion
    condition:
      field: message
      operator: matches
      value: "(?i)(\\\\b(delete|remove|deactivate|disable|terminate)\\\\b.{0,80}\\\\b(user|employee|account|profile|E\\\\d{3,})\\\\b|\\\\b(user|employee|account|profile|E\\\\d{3,})\\\\b.{0,80}\\\\b(delete|remove|deactivate|disable|terminate)\\\\b)"
    action: deny
    message: "User deletion and deactivation requests are blocked by HR policy."
    priority: 950

  - name: block-third-party-salary-lookup
    condition:
      field: message
      operator: matches
      value: "(?i)(?=.*\\\\b(salary|compensation|pay|bonus|paycheck)\\\\b)(?=.*\\\\b(other|coworker|co-worker|colleague|teammate|someone\\\\s+else|another\\\\s+employee|employee\\\\s+[A-Za-z]?\\\\d+|my\\\\s+manager|my\\\\s+boss|my\\\\s+report)\\\\b).*"
    action: deny
    message: "Disclosing another employee's compensation is not permitted by HR policy."
    priority: 900

  - name: audit-benefits-and-pto
    condition:
      field: message
      operator: matches
      value: "(?i)(pto|paid\\\\s+time\\\\s+off|leave|holiday|benefits?|enrollment|401k|medical|dental|vision)"
    action: audit
    message: "HR self-service query logged for compliance trail."
    priority: 100
"""
(policies_dir / "hr-policy.yaml").write_text(hr_policy_yaml, encoding="utf-8")
utils.print_ok("Created hosted-agent/policies/hr-policy.yaml")

# ---- agent.yaml ----
agent_yaml = f"""\
kind: hosted
name: {hosted_agent_name}
protocols:
  - protocol: responses
    version: 1.0.0
resources:
  cpu: "1"
  memory: 2Gi
environment_variables:
  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
    value: {hosted_agent_model}
  - name: ENABLE_INSTRUMENTATION
    value: "true"
  - name: ENABLE_SENSITIVE_DATA
    value: "true"
  - name: OTEL_SERVICE_NAME
    value: {hosted_agent_name}
  - name: OTEL_RESOURCE_ATTRIBUTES
    value: service.name={hosted_agent_name},service.namespace=citadel,deployment.environment=workshop
"""
(agent_dir / "agent.yaml").write_text(agent_yaml, encoding="utf-8")
utils.print_ok("Created hosted-agent/agent.yaml")

utils.print_info(f"\nAll agent source files written to: {agent_dir.resolve()}")

<a id='3'></a>
### 3️⃣ Build & Push Container Image

Uses `az acr build` to build the Docker image in the cloud and push it to ACR. **No local Docker installation required.**

> The signed-in user needs **Contributor** (or at minimum an ACR task-scheduling permission) on the ACR. If you deployed the spoke with `deploy-spoke-foundry`, you already have Owner on the spoke RG which inherits to the ACR.

In [ ]:
# NOTE: --no-logs is required because Azure CLI on Windows crashes streaming
# build logs due to a cp1252 encoding bug (pip output contains Unicode chars
# that colorama can't encode). The build runs server-side regardless.
acr_build_cmd = (
    f'az acr build'
    f' --registry {env_acr_name}'
    f' --resource-group {env_foundry_resource_group}'
    f' --image {hosted_agent_name}:{hosted_agent_image_tag}'
    f' --file hosted-agent/Dockerfile'
    f' --no-logs'
    f' hosted-agent/'
)

utils.print_info(f"Building image in ACR: {hosted_agent_image}")
utils.print_info("Build logs suppressed (Windows encoding bug). Polling for completion...")

output = utils.run(acr_build_cmd, "ACR build queued", "ACR build queue failed")

# Wait for the image to appear in ACR (build takes ~2-3 min)
import time as _t
for attempt in range(24):  # up to 4 minutes
    verify = utils.run(
        f'az acr repository show-tags --name {env_acr_name} --repository {hosted_agent_name} --output json',
        "", ""
    )
    if verify.success and verify.json_data and hosted_agent_image_tag in verify.json_data:
        utils.print_ok(f"Image verified in ACR: {hosted_agent_image}")
        break
    utils.print_info(f"  Waiting for image... ({(attempt+1)*10}s)")
    _t.sleep(10)
else:
    utils.print_error(f"Image '{hosted_agent_image_tag}' not found after 4 min. Check ACR build logs in the portal.")

<a id='4'></a>
### 4️⃣ Deploy Hosted Agent to Foundry

Creates a new hosted agent version in the Foundry project using the Python SDK. The agent is deployed with the container image from ACR and configured to use the BYO Gateway connection for LLM access.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential

utils.print_info(f"Connecting to Foundry endpoint: {foundry_project_endpoint}")

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=credential,
    allow_preview=True,
)
utils.print_ok(f"Foundry project client initialized for: {env_foundry_project_name}")

# Deploy the hosted agent
utils.print_info(f"Deploying hosted agent: {hosted_agent_name}")
utils.print_info(f"Image: {hosted_agent_image}")
utils.print_info(f"Model: {hosted_agent_model}")

agent = project_client.agents.create_version(
    agent_name=hosted_agent_name,
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(
                protocol=AgentProtocol.RESPONSES,
                version="1.0.0",
            )
        ],
        cpu="1",
        memory="2Gi",
        image=hosted_agent_image,
        environment_variables={
            "AZURE_AI_MODEL_DEPLOYMENT_NAME": hosted_agent_model,
            "ENABLE_INSTRUMENTATION": "true",
            "ENABLE_SENSITIVE_DATA": "true",
            "OTEL_SERVICE_NAME": hosted_agent_name,
            "OTEL_RESOURCE_ATTRIBUTES": (
                f"service.name={hosted_agent_name},"
                "service.namespace=citadel,"
                "deployment.environment=workshop"
            ),
            # Unlocks GenAI prompt/completion/token attributes on model spans
            # (otherwise azure.ai.projects.telemetry logs a WARNING at startup
            # and agent365_exporter reports "No eligible genAI spans to export").
        },
    ),
)

utils.print_ok(f"Agent version created: {agent.name} (version: {agent.version})")
utils.print_info(f"Status: {agent.status}")

<a id='5'></a>
### 5️⃣ Wait for Agent to Become Active

Poll the agent version status until it becomes `active`. This typically takes 2-5 minutes as the platform pulls the image and starts the container.

In [ ]:
max_wait_seconds = 600  # 10 minutes
poll_interval = 15

elapsed = 0
while elapsed < max_wait_seconds:
    version = project_client.agents.get_version(
        agent_name=agent.name,
        agent_version=agent.version,
    )
    status = version.status
    utils.print_info(f"[{elapsed}s] Agent status: {status}")

    if status == "active":
        utils.print_ok(f"Agent is active! (took ~{elapsed}s)")
        break
    elif status in ("failed", "error"):
        utils.print_error(f"Agent deployment failed with status: {status}")
        break

    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    utils.print_error(f"Timed out after {max_wait_seconds}s waiting for agent to become active.")

<a id='5a'></a>
### 5a. Assign RBAC to Agent Identity

Every hosted agent gets a dedicated Microsoft Entra ID (agent identity) at deploy time. This identity needs the **Foundry User** role on the Foundry account so it can access conversation storage and model endpoints.

> When deploying with `azd`, this role is assigned automatically. Since we're using the SDK, we assign it explicitly here.

In [ ]:
# Retrieve the agent identity assigned by the platform
agent_identity = version.instance_identity

if agent_identity is None:
    # Fallback: fetch from agent-level details
    agent_details = project_client.agents.get(agent_name=hosted_agent_name)
    agent_identity = agent_details.instance_identity

if agent_identity is None:
    utils.print_error("Could not resolve agent identity. Ensure the agent is in 'active' state.")
else:
    agent_principal_id = agent_identity.principal_id
    utils.print_ok(f"Agent identity principal ID: {agent_principal_id}")

    foundry_account_scope = (
        f"/subscriptions/{env_foundry_subscription_id}"
        f"/resourceGroups/{env_foundry_resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{env_foundry_account_name}"
    )

    # Check if Foundry User role is already assigned
    check = subprocess.run(
        ["az", "role", "assignment", "list",
         "--assignee-object-id", agent_principal_id,
         "--role", "Foundry User",
         "--scope", foundry_account_scope,
         "--query", "[0].id", "-o", "tsv"],
        capture_output=True, text=True, shell=True,
    )

    if check.stdout.strip():
        utils.print_ok("Foundry User role is already assigned.")
    else:
        utils.print_info(f"Assigning 'Foundry User' to agent identity on: {env_foundry_account_name}")
        result = subprocess.run(
            ["az", "role", "assignment", "create",
             "--assignee-object-id", agent_principal_id,
             "--assignee-principal-type", "ServicePrincipal",
             "--role", "Foundry User",
             "--scope", foundry_account_scope,
             "--only-show-errors", "-o", "none"],
            capture_output=True, text=True, shell=True,
        )
        if result.returncode == 0:
            utils.print_ok("Foundry User role assigned. Waiting 60s for RBAC propagation...")
            time.sleep(60)
            utils.print_ok("Done — RBAC should be active now.")
        else:
            utils.print_error(f"Failed to assign role: {result.stderr}")

<a id='6'></a>
### 6️⃣ Invoke the Hosted Agent

Use `project.get_openai_client(agent_name=...)` to invoke the hosted agent via the Responses API. This sends a request to the running agent container.

In [ ]:
import uuid
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

# Get an OpenAI client scoped to the hosted agent.
# Use an explicit Azure CLI credential with a longer timeout because
# Azure Identity's default 10s CLI subprocess timeout can be too short
# on Windows when refreshing the https://ai.azure.com token.
invoke_project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=AzureCliCredential(process_timeout=60),
    allow_preview=True,
)
openai_client = invoke_project_client.get_openai_client(agent_name=hosted_agent_name)

# MAF's ResponsesHostServer emits response items in shapes the OpenAI SDK's
# `response.output_text` convenience aggregator doesn't recognize. Dump the
# response to a dict and walk every nested value looking for text under
# common keys (text, content, output_text, value, message, delta).
def extract_text(response) -> str:
    # Surface server-side failures rather than silently returning ""
    err = getattr(response, "error", None)
    status = getattr(response, "status", None)
    if err is not None:
        code = getattr(err, "code", None) or (err.get("code") if isinstance(err, dict) else None)
        msg = getattr(err, "message", None) or (err.get("message") if isinstance(err, dict) else None)
        return f"[agent error: status={status} code={code} message={msg}]"
    text = getattr(response, "output_text", None)
    if text:
        return text
    try:
        data = response.model_dump()
    except Exception:
        data = getattr(response, "__dict__", {}) or {}

    chunks: list[str] = []
    text_keys = ("text", "output_text", "value", "message", "delta")

    def walk(node):
        if isinstance(node, dict):
            for key in text_keys:
                v = node.get(key)
                if isinstance(v, str) and v.strip():
                    chunks.append(v)
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for item in node:
                walk(item)

    walk(data)

    seen, unique = set(), []
    for c in chunks:
        c = c.strip()
        if c and c not in seen:
            seen.add(c)
            unique.append(c)
    return "\n".join(unique)


# Use a conversation_id so Foundry Traces groups requests together
conversation_id = str(uuid.uuid4())
utils.print_info(f"Conversation ID: {conversation_id}")

# Test 1: PTO balance
utils.print_info("Sending: How many PTO days do I have left? My employee ID is E12345.")
response = openai_client.responses.create(
    input="How many PTO days do I have left? My employee ID is E12345.",
    metadata={"conversation_id": conversation_id},
)

utils.print_ok(f"Agent response: {extract_text(response)}")
print(f"\nResponse ID: {response.id}")

# Test 2: Benefits summary
utils.print_info("\nSending: Give me a quick summary of the 401k plan.")
response2 = openai_client.responses.create(
    input="Give me a quick summary of the 401k plan.",
    metadata={"conversation_id": conversation_id},
)

utils.print_ok(f"Agent response: {extract_text(response2)}")

<a id='6a'></a>
### 6a. Test AGT Governance Enforcement

Send five prompts that exercise the AGT policy bundled with the container (`policies/hr-policy.yaml`):

1. **Allowed** — PTO lookup by employee ID (tool-grounded answer).
2. **Audited** — Benefits summary (succeeds, emitted through AGT OTel as policy telemetry).
3. **Denied (PII / SSN)** — Request containing an SSN; policy blocks it.
4. **Denied (3rd-party salary)** — Request for a coworker's salary; policy blocks it.
5. **Denied (`delete_user`)** — Request that would require the destructive `delete_user` tool; policy blocks it based on the message before the tool can run.

When a request is denied, the agent returns a message starting with `⛔ Policy violation:` (set by `GovernancePolicyMiddleware` via `MiddlewareTermination`). The denial may also surface as an HTTP error from the Responses API depending on how the host serializes the terminated response — both outcomes are caught below.

In [ ]:
governance_tests = [
    ("Allowed (PTO)",   "How many PTO days do I have left? My employee ID is E12345."),
    ("Audited (401k)",  "Tell me about my 401k benefits."),
    ("Denied (SSN)",    "My coworker's SSN is 123-45-6789, look up their leave balance."),
    ("Denied (salary)", "What is my colleague's salary?"),
    ("Denied (delete_user)", "Delete user E12345 from Contoso HR."),
]

governance_conv_id = str(uuid.uuid4())
utils.print_info(f"Governance Conversation ID: {governance_conv_id}")

for label, prompt in governance_tests:
    utils.print_info(f"\n--- {label} ---")
    utils.print_info(f"User: {prompt}")
    try:
        r = openai_client.responses.create(
            input=prompt,
            metadata={"conversation_id": governance_conv_id},
        )
        text = extract_text(r)
        if "Policy violation" in text:
            utils.print_warning(f"BLOCKED → {text}")
        else:
            utils.print_ok(f"Allowed → {text}")
    except Exception as e:
        utils.print_warning(f"BLOCKED via exception → {type(e).__name__}: {e}")

<a id='7'></a>
### 7️⃣ Multi-Turn Conversation

Test a multi-turn conversation with the hosted agent using `previous_response_id` to maintain context across turns.

In [ ]:
conversation_messages = [
    "What are the next Contoso holidays in the US?",
    "What about in Japan?",
    "And when does open enrollment start?",
]

previous_response_id = None
multi_turn_conversation_id = str(uuid.uuid4())
utils.print_info(f"Conversation ID: {multi_turn_conversation_id}")

for i, message in enumerate(conversation_messages, 1):
    utils.print_info(f"\n--- Turn {i} ---")
    utils.print_info(f"User: {message}")

    kwargs = {
        "input": message,
        "metadata": {"conversation_id": multi_turn_conversation_id},
    }
    if previous_response_id:
        kwargs["previous_response_id"] = previous_response_id

    response = openai_client.responses.create(**kwargs)
    previous_response_id = response.id

    utils.print_ok(f"Agent: {extract_text(response)}")

utils.print_info(f"\nConversation completed ({len(conversation_messages)} turns)")

<a id='8'></a>
### 8️⃣ Streaming Responses

Re-run the same queries with `stream=True` to demonstrate **streaming mode**. Tokens are printed as they arrive, significantly improving perceived latency (time-to-first-token) — the total inference time is similar, but the user sees output within seconds instead of waiting for the full response.

In [ ]:
# Streaming: single-turn queries
streaming_conversation_id = str(uuid.uuid4())
utils.print_info(f"Streaming Conversation ID: {streaming_conversation_id}")

# Test 1: PTO balance (streaming)
utils.print_info("Sending: How many PTO days do I have left? My employee ID is E67890.")
stream = openai_client.responses.create(
    input="How many PTO days do I have left? My employee ID is E67890.",
    metadata={"conversation_id": streaming_conversation_id},
    stream=True,
)
response = None
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        response = event.response
print()
utils.print_ok("Stream complete")
print(f"Response ID: {response.id}")

# Test 2: Benefits summary (streaming)
utils.print_info("\nSending: Summarize the medical and dental plans for me.")
stream2 = openai_client.responses.create(
    input="Summarize the medical and dental plans for me.",
    metadata={"conversation_id": streaming_conversation_id},
    stream=True,
)
response2 = None
for event in stream2:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        response2 = event.response
print()
utils.print_ok("Stream complete")

In [ ]:
# Streaming: multi-turn conversation
conversation_messages = [
    "What are the next Contoso holidays in the UK?",
    "Great. Now summarize the vision plan.",
    "And how much PTO does employee E24680 have left?",
]

previous_response_id = None
streaming_multi_turn_id = str(uuid.uuid4())
utils.print_info(f"Streaming Conversation ID: {streaming_multi_turn_id}")

for i, message in enumerate(conversation_messages, 1):
    utils.print_info(f"\n--- Turn {i} ---")
    utils.print_info(f"User: {message}")

    kwargs = {
        "input": message,
        "metadata": {"conversation_id": streaming_multi_turn_id},
        "stream": True,
    }
    if previous_response_id:
        kwargs["previous_response_id"] = previous_response_id

    stream = openai_client.responses.create(**kwargs)
    response = None
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif event.type == "response.completed":
            response = event.response
    print()
    previous_response_id = response.id

    utils.print_ok(f"Turn {i} complete")

utils.print_info(f"\nStreaming conversation completed ({len(conversation_messages)} turns)")

<a id='9'></a>
### 9️⃣ Cleanup

Delete the hosted agent entirely (all versions + agent identity). Set `cleanup_enabled = True` to remove the agent.

> **Note:** This removes the agent and its Entra ID identity from the Foundry project. The ACR image, RBAC role assignments, Foundry project, and other spoke resources are **not** affected. Orphaned RBAC assignments (for the deleted identity) are harmless and ignored by Azure.

In [ ]:
cleanup_enabled = False  # Set to True to enable cleanup of the agent after testing

if cleanup_enabled:
    try:
        result = project_client.agents.delete(agent_name=agent.name)
        utils.print_ok(f"Agent deleted: {agent.name} (deleted={result.get('deleted', True)})")
    except Exception as e:
        utils.print_warning(f"Failed to delete agent: {e}")
else:
    utils.print_info(f"Cleanup is disabled. Set cleanup_enabled = True to remove the agent.")
    utils.print_info(f"Agent retained: {agent.name} (version: {agent.version})")

## View AGT audit results in Application Insights

AGT policy and audit decisions are exported as `agt.audit.event` trace records in Application Insights. To inspect denied policy decisions, open the Application Insights resource connected to the Foundry account, go to **Logs**, and run this KQL query:

```kusto
traces
| where timestamp > ago(30m)
| where message == "agt.audit.event"
| extend cd = tostring(customDimensions)
| extend ce = tostring(customDimensions["agt.cloudevent"])
| where cd has "ai.agentmesh.policy.violation"
| extend reason_double = extract(@"'reason':\s*\\?""(.*?)\\?"",\s*'message_preview'", 1, ce)
| extend reason_single = extract(@"'reason':\s*'(.*?)',\s*'message_preview'", 1, ce)
| extend reason = iff(isnotempty(reason_double), reason_double, reason_single)
| extend matched_rule = extract(@"'matched_rule':\s*'([^']*)'", 1, ce)
| extend message_preview_double = extract(@"'message_preview':\s*\\?""(.*?)\\?""", 1, ce)
| extend message_preview_single = extract(@"'message_preview':\s*'(.*?)'", 1, ce)
| extend message_preview = iff(isnotempty(message_preview_double), message_preview_double, message_preview_single)
| project timestamp, reason, matched_rule, message_preview, message, customDimensions
| order by timestamp desc
```

These records appear in the App Insights `traces` table rather than as separate AGT spans in the Foundry trace tree.
